In [1]:
!pip install polars

  Using cached polars-1.38.1-py3-none-any.whl.metadata (10 kB)
  Using cached polars_runtime_32-1.38.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.5 kB)
Using cached polars-1.38.1-py3-none-any.whl (810 kB)
Using cached polars_runtime_32-1.38.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (45.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]


In [5]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import math
import pandas as pd
import numpy as np

## Read data from S3 and join

In [ ]:
source_bucket = 's3://msds-26.2-data'
file_names = ['sanitized_2022.csv', 'sanitized_2023.csv', 'sanitized_2024.csv', 'sanitized_2025.csv']

In [ ]:
# Read csv from s3 and concat
df_recovery = pl.DataFrame()
for name in file_names:
    file_path = f'{source_bucket}/{name}'
    df_name = pl.read_csv(file_path, separator="\t")
    df_recovery = pl.concat([df_recovery, df_name])

In [ ]:
del df_name

In [ ]:
len(df_recovery)

In [ ]:
pl.Config.set_tbl_rows(50)

In [ ]:
df_recovery.sort(by=['hashed_fc', 'year', 'month', 'week', 'gl_product_group'])

## Data Cleaning

In [ ]:
# Drop columns where all values are null
cols_to_drop = (
    df_recovery.select(pl.all().is_null().all())
    .unpivot()
    .filter(pl.col("value") == True)
    .select("variable")
    .to_series()
    .to_list()
)

# Drop the identified columns from the original DataFrame
df_recovery = df_recovery.drop(cols_to_drop)

In [ ]:
# Remove all C-Returns from the data
df_recovery = df_recovery.filter(pl.col('recovery_type') != 'C-Returns')

In [ ]:
# Create target variable for propensity to waste model
df_recovery = df_recovery.with_columns(
    pl.when(pl.col('recovery_type') == 'Sales').then(pl.lit(0))
    .otherwise(pl.lit(1)).alias('recovery')
)

In [ ]:
df_recovery.filter(pl.col('country') != pl.col('marketplace_id'))

In [ ]:
# Remove marketplace_id b/c it's a duplicate of country
df_recovery = df_recovery.drop(['marketplace_id'])

In [ ]:
# Reorder the columns
df_recovery = df_recovery.select([
 'hashed_fc',
 'year',
 'month',
 'week',
 'gl_product_group',
 'product_type',
 'macro_category',
 'item_disposition_code',
 'reason_code',
 'application_name',
 'is_stranded',
 'reason_code_type',
 'reason_code_stranded',
 'stranded_potential_issue',
 'is_hazmat',
 'units',
 'cogs',
 'weight',
 'country',
 'country_state',
 'zip_code',
 'site_type',
 'site_category',
 'recovery',
 'recovery_type'])

In [ ]:
destination = "s3://msds-26.2-data/clean/combined_recovery_data.parquet"
df_recovery.write_parquet(destination)

## Aggregation and Feature Engineering

In [2]:
df_recovery = pl.read_parquet("s3://msds-26.2-data/clean/combined_recovery_data.parquet")

In [3]:
# Remove 3000 isntances of missing gl_product_group
df_recovery = df_recovery.filter(pl.col('gl_product_group').is_not_null())

In [4]:
# Aggregate to the site-week-gl level
df_recovery_agg = (
    df_recovery
    .group_by(['hashed_fc', 'year', 'month', 'week', 'gl_product_group'])
    .agg(
        pl.len().alias('num_records'),

        pl.when(pl.col("macro_category") == "RETAIL")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_RETAIL"),

        pl.when(pl.col("macro_category") == "FBA")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_FBA"),

        pl.when(pl.col("is_hazmat") == "Y")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_hazmat"),

        pl.when(pl.col("product_type") == "Food")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_food"),

        pl.when(pl.col("product_type") == "Non Food")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_non_food"),

        pl.when(pl.col("product_type") == "Pet Food")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_pet_food"),

        pl.col('units').sum().alias('units_total'),
        pl.col('cogs').sum().alias('cogs_total'),
        pl.col('weight').sum().alias('weight_total'),

        pl.first('country'),
        pl.first('country_state'),
        pl.first('zip_code'),
        pl.first('site_type'),
        pl.first('site_category'),

        pl.when(pl.col("recovery_type") != "Sales")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_recovered"),
    )
)

In [5]:
# Derive cogs per unit features
df_recovery_agg = df_recovery_agg.with_columns([
    (pl.col("cogs_total") / 
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("avg_cogs_per_unit"),

    (pl.col("weight_total") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("avg_weight_per_unit"),

    (pl.col("cogs_total") /
     pl.when(pl.col("weight_total") > 0)
       .then(pl.col("weight_total"))
       .otherwise(None)
    ).alias("cogs_per_unit_weight"),
])

In [6]:
# Derive share features
df_recovery_agg = df_recovery_agg.with_columns([
    (pl.col("units_food") / 
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_food"),

    (pl.col("units_non_food") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_non_food"),

    (pl.col("units_pet_food") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_pet_food"),

    (pl.col("units_RETAIL") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_RETAIL"),

    (pl.col("units_FBA") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_FBA"),

    (pl.col("units_hazmat") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_hazmat"),
])

In [8]:
# Create ISO week-date
df_recovery_agg = df_recovery_agg.with_columns(
    pl.date(pl.col("year"), 1, 4)
    .dt.truncate("1w")
    .dt.offset_by(
        (pl.col("week").cast(pl.Int32) - 1).cast(pl.Utf8) + "w"
    )
    .alias("week_date")
)

In [9]:
group_keys = ["hashed_fc", "gl_product_group"]

# Build complete week grid
full_weeks = (
    df_recovery_agg.group_by(group_keys)
      .agg([
          pl.min("week_date").alias("start"),
          pl.max("week_date").alias("end"),
      ])
      .with_columns(
          pl.date_ranges("start", "end", interval="1w").alias("week_date")
      )
      .explode("week_date")
)

# Join and Sort
df_recovery_agg_full_weeks = (
    full_weeks
    .join(df_recovery_agg, on=group_keys + ["week_date"], how="left")
    .sort(group_keys + ["week_date"])
)

In [10]:
# Fill missing week for lag feature creation
df_recovery_agg_full_weeks = (
    df_recovery_agg_full_weeks
    .sort(["hashed_fc", "gl_product_group",  "week_date"])
    .with_columns([
        pl.col("units_total").fill_null(0),
        pl.col("units_recovered").fill_null(0),
        pl.col("avg_cogs_per_unit").fill_null(0),
        pl.col("avg_weight_per_unit").fill_null(0),
        pl.col("cogs_per_unit_weight").fill_null(0),
        
        pl.col("units_RETAIL").fill_null(0),
        pl.col("units_FBA").fill_null(0),
        pl.col("units_food").fill_null(0),
        pl.col("units_non_food").fill_null(0),
        pl.col("units_pet_food").fill_null(0),
        pl.col("units_hazmat").fill_null(0),
        
        pl.col("share_RETAIL").fill_null(0),
        pl.col("share_FBA").fill_null(0),
        pl.col("share_food").fill_null(0),
        pl.col("share_non_food").fill_null(0),
        pl.col("share_pet_food").fill_null(0),
        pl.col("share_hazmat").fill_null(0),
        
    ])
)

In [11]:
# Create Lag features
df_recovery_agg_full_weeks = df_recovery_agg_full_weeks.with_columns(
    # Lag features for units_recovered
    pl.col('units_recovered').shift(1).over(group_keys).alias('units_recovered_lag_1'),
    pl.col('units_recovered').shift(2).over(group_keys).alias('units_recovered_lag_2'),
    pl.col('units_recovered').shift(4).over(group_keys).alias('units_recovered_lag_4'),

    # Rolling mean for units_recovered
    pl.col("units_recovered")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("units_recovered_rolling_mean_4"),
    
    pl.col("units_recovered")
        .shift(1)
        .rolling_std(4)
        .over(group_keys)
        .alias("units_recovered_rolling_std_4"),
    
    pl.col("units_recovered")
        .shift(1)
        .rolling_mean(12)
        .over(group_keys)
        .alias("units_recovered_rolling_mean_12"),

    # Lagged mix features
    pl.col("avg_cogs_per_unit").shift(1).over(group_keys).alias("avg_cogs_per_unit_lag_1"),
    pl.col("avg_weight_per_unit").shift(1).over(group_keys).alias("avg_weight_per_unit_lag_1"),

    pl.col("avg_cogs_per_unit")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("avg_cogs_per_unit_mean_4"),

    pl.col("avg_weight_per_unit")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("avg_weight_per_unit_mean4"),

    # Volume lags
    pl.col("units_total").shift(1).over(group_keys).alias("units_lag_1"),

    pl.col("units_total")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("units_mean_4"),

    pl.col("units_RETAIL").shift(1).over(group_keys).alias("units_RETAIL_lag_1"),

    pl.col("units_RETAIL")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("units_RETAIL_mean_4"),

    pl.col("units_FBA").shift(1).over(group_keys).alias("units_FBA_lag_1"),

    pl.col("units_FBA")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("units_FBA_mean_4"),

    pl.col("units_hazmat").shift(1).over(group_keys).alias("units_hazmat_lag_1"),

    pl.col("units_hazmat")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("units_hazmat_mean_4"),

    pl.col("units_food").shift(1).over(group_keys).alias("units_food_lag_1"),

    pl.col("units_food")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("units_food_mean_4"),

    pl.col("units_non_food").shift(1).over(group_keys).alias("units_non_food_lag_1"),
    
    pl.col("units_non_food")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("units_non_food_mean_4"),

    pl.col("units_pet_food").shift(1).over(group_keys).alias("units_pet_food_lag_1"),

    pl.col("units_pet_food")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("units_pet_food_mean_4"),

    # Share lag
    pl.col("share_RETAIL").shift(1).over(group_keys).alias("share_RETAIL_lag_1"),

    pl.col("share_RETAIL")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("share_RETAIL_mean_4"),

    pl.col("share_FBA").shift(1).over(group_keys).alias("share_FBA_lag_1"),

    pl.col("share_FBA")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("share_FBA_mean_4"),

    pl.col("share_hazmat").shift(1).over(group_keys).alias("share_hazmat_lag_1"),

    pl.col("share_hazmat")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("share_hazmat_mean_4"),

    pl.col("share_food").shift(1).over(group_keys).alias("share_food_lag_1"),

    pl.col("share_food")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("share_food_mean_4"),

    pl.col("share_non_food").shift(1).over(group_keys).alias("share_non_food_lag_1"),
    
    pl.col("share_non_food")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("share_non_food_mean_4"),

    pl.col("share_pet_food").shift(1).over(group_keys).alias("share_pet_food_lag_1"),

    pl.col("share_pet_food")
        .shift(1)
        .rolling_mean(4)
        .over(group_keys)
        .alias("share_pet_food_mean_4"),
    
)

In [12]:
# Drop buffer weeks
df_recovery_agg_full_weeks = df_recovery_agg_full_weeks.drop_nulls(subset=['week'])

In [3]:
df_recovery_agg_full_weeks = pl.read_parquet("s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_new_features.parquet")

In [6]:
# Create sine and cosine transformation for week
period = 52.1775
df_recovery_agg_full_weeks = df_recovery_agg_full_weeks.with_columns(
    (np.sin(2 * np.pi * pl.col('week') / period)).alias('week_sin'),
    (np.cos(2 * np.pi * pl.col('week') / period)).alias('week_cos')
)

In [8]:
destination = "s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_new_features.parquet"
df_recovery_agg_full_weeks.write_parquet(destination)